In [15]:
import librosa
import numpy as np
from scipy.fft import dct
from tensorflow.keras import layers,models
from pathlib import Path


### Je vais d'abord effectuer l'opération du MFCC sur un audio pour bien comprendre le mmécanisme avant de réaliser une pipeline complète sur tous mes enregistrements audios.

In [5]:
## signal est un tableau qui represente les amplitudes, sr est la fréquence d'échantillonage
signal,sr = librosa.load("03-01-01-01-01-01-01.wav",sr = 16000)
print(signal.shape)

(52853,)


### Pour une fréquence d'échantillonage de 16kHz et 52853 amplitudes mon audio fait approximativement 52853/16000 = 3.3033 secondes

In [6]:
## On fait un découpage en frame
## On utilise une fenetre de 25ms et un décalage de 10ms
## Sachant que je fais un découpage de 25ms et que chaque amplitude est mesurée sur 16000 Hz, chaque frame doit contenir 0.025*16000 = 400 echantillons, d'où frame_lenght = 400
## et le decalage de 10 ms correspond à 16000*0.01 = 160 échantillons
frames = librosa.util.frame(signal,frame_length=400,hop_length=160)
print(frames.shape)

(400, 328)


### J'ai donc un tableau de 328 frames où chaque frame contient 400 échantillons

In [7]:
## On va maintenant passer du domaine temporel au domaine fréquentielle
## C'est à dire que pour chaque frame, on va voir à quel point elle est composée de plusieurs fréquences(0Hz,40Hz,...)
## Pour cela on applique la transformée de fourier rapide
## la rfft ne retourne que 400/2 +1 = 201 valeurs pour chaque frame de 400 échantillons(que les valeurs positives)
fft = np.fft.rfft(frames.T)
print(fft.shape)
## Je retranspose pour avoir les dimensions (valeurs_fft,echantillons)
fft = fft.T
print(fft.shape)

(328, 201)
(201, 328)


In [8]:
## fft est une matrice de complexes
print(fft[0,0])

(4.0748233e-05+0j)


In [9]:
## On calcule desormais le spectre, qui represente le module de chaque fréquence
spectre = np.abs(fft)**2
print(spectre.shape)
print(spectre[0,0])

(201, 328)
1.6604185e-09


### J'ai maintenant un spectogramme. C'est à dire une matrice où chaque ligne correspond à une fréquence et chaque colonne à un intensité du signal

In [10]:
### L'oreille humaine ne distingue pas les fréquences de la même façon. On est plus sensible aux basses fréquences qu'aux grandes fréquences
### On va appliquer un filtre de Mel, c'est à dire regrouper les fréquences par plages.
## sr représente la fréquence d'échantillonage
## n_fft la taille de la fft
## n_mels représente le nombre de bandes de mel que je veux obtenir
filtres = librosa.filters.mel(sr=16000,n_fft=400,n_mels=40)
print(filtres.shape)

(40, 201)


In [11]:
## j'applique ce fitre à mon spectre calculé plus haut
mel = filtres@spectre
print(mel.shape)
print(mel[0,0])

(40, 328)
4.0218814e-11


In [12]:
## Je calcule désormais le logarithme, pour d'abord diminuer les valeurs numériques
## Ensuite modéliser le fait que on ne perçoit pas les sons de façon linéaire.
## une même phrase répétée plus ne dit pas seulement que le volume a augmenté, ça traduit des émotions aussi.
log_mel = np.log(mel + 1e-10) ## 1e-10 évite d'avoir log(0)
print(log_mel.shape)
print(log_mel[0,0])

(40, 328)
-22.687817


In [13]:
## Les bandes proches contiennent des informations qui se ressemblent. Et pour éviter d'avoir des données beaucoup corrélées on utilise la dct
## la dct(transformation en cosinus discret) permet de ranger l'information de façon pertinente. Elle range les valeurs qui on le plus d'importance au début et relegue le reste à la fin.
mfcc = dct(log_mel,type=2,norm='ortho')
## type = 2 indique une formule standard utilisée pour les traitements de signal
## et nom = 'ortho' sert à normaliser les données pour qu'il reste dans une certaine plage
print(mfcc.shape)
print(mfcc[0,0])
print(mfcc[39,327])

(40, 328)
-257.13565
-0.64156604


### Maintenant je peux retrouver ces caractéristiques mfcc juste en appliquant _librosa.features.mfcc_

In [19]:
## Je recupere sous forme de dictionnaire, le chemin, l'acteur, et l'émotion
dossier = Path("archive")
train_files = []
test_files = []
for chemin_acteur in dossier.iterdir():
    if not chemin_acteur.is_dir():
        continue
    acteur = int(chemin_acteur.name.split("_")[1])

    ## On parcourt les fichiers audios maintenant
    for audio in chemin_acteur.glob("*.wav"):
        ## l'émotion est la troisieme valeur du nom
        emotion = audio.stem.split("-")[2]
        données = {"chemin":str(audio),
                   "emotion":emotion,
                   "actor":acteur}
        
        if acteur <=20:
            train_files.append(données)
        else:
            test_files.append(données)

In [22]:
print(f"Train : {len(train_files)} fichiers d'entrainement")
print(f"Test : {len(test_files)} fichiers de test")


Train : 1200 fichiers d'entrainement
Test : 240 fichiers de test


In [21]:
train_files[0]

{'chemin': 'archive\\Actor_01\\03-01-01-01-01-01-01.wav',
 'emotion': '01',
 'actor': 1}

In [125]:
mfcc_train = []
emotion_train = []
mfcc_test = []
emotion_test = []
for i in train_files:
    signal_train,sr = librosa.load(i["chemin"],sr=16000)
    res_train = librosa.feature.mfcc(y=signal_train,
                               sr=16000,
                               n_mfcc=40,
                               n_fft=400,
                               hop_length=160)
    mfcc_train.append(res_train)
    emotion_train.append(i["emotion"])
    
for j in test_files:
    signal_test,sr = librosa.load(j["chemin"],sr = 16000)
    res_test = librosa.feature.mfcc(y=signal_test,
                                   sr=16000,
                                   n_mfcc=40,
                                   n_fft=400, 
                                   hop_length=160)
    mfcc_test.append(res_test)
    emotion_test.append(j["emotion"])
    

In [126]:
print(mfcc_train[5].shape)
print(mfcc_train[100].shape)
## je n'ai pas les mêmes dimensions, je vais donc réaliser un padding

(40, 361)
(40, 381)


In [127]:
long_max = 350

# Fonction pour appliquer le padding à une liste de tableaux
def pad_liste(liste_mfcc, max_len):
    liste_paddee = []
    for mfcc in liste_mfcc:
        # On récupère la longueur actuelle (nombre de colonnes)
        current_len = mfcc.shape[1] 
        
        if current_len < max_len:
            pad = max_len - current_len
            # On applique le padding (mode constant met des zéros)
            # ((0,0), (0,pad)) signifie : pas de padding sur les MFCC, padding sur le temps
            mfcc_padde = np.pad(mfcc, ((0, 0), (0, pad)), mode="constant")
            liste_paddee.append(mfcc_padde)
        else:
            # Si le fichier est déjà trop long, on peut le tronquer ou le garder
            liste_paddee.append(mfcc[:, :max_len])
            
    return np.array(liste_paddee)

# Application sur mes données
mfcc_train = pad_liste(mfcc_train, long_max)
mfcc_test = pad_liste(mfcc_test, long_max)

In [128]:
emotion_test = np.array(emotion_test)
emotion_train = np.array(emotion_train)

In [129]:
print(f"mfcc_train: {mfcc_train.shape}")
print(f"emotion_train: {emotion_train.shape}")
print(f"mfcc_test: {mfcc_test.shape}")
print(f"emotion_test: {emotion_test.shape}")


mfcc_train: (1200, 40, 350)
emotion_train: (1200,)
mfcc_test: (240, 40, 350)
emotion_test: (240,)


In [131]:
## Mes données sont organisées en (nbre_exemples,n_mfcc,temps), il manque une dimension de canal pour appliquer les filtres convolutifs de keras
## Je vais donc les ajouter

mfcc_train = mfcc_train.reshape(1200,40,350,1)
mfcc_test = mfcc_test.reshape(240,40,350,1)
print(f"mfcc_train: {mfcc_train.shape}")
print(f"mfcc_test: {mfcc_test.shape}")

mfcc_train: (1200, 40, 350, 1)
mfcc_test: (240, 40, 350, 1)


In [132]:
## je vais normaliser mes données
xmin = mfcc_train.min()
xmax = mfcc_train.max()
mfcc_train = (mfcc_train - xmin)/(xmax-xmin)
mfcc_test = (mfcc_test - xmin)/(xmax-xmin)

In [133]:
print(emotion_train[100])

06


In [134]:
## je transforme mes données d'émotions de string a int pour pouvoir entrainer mon modele.
emotion_train = emotion_train.astype(int)
emotion_test = emotion_test.astype(int)
np.unique(emotion_train)

array([1, 2, 3, 4, 5, 6, 7, 8])

In [135]:
## La fonction crossentropy attend des labels qui vont de 0 à 7 et pas de 1 à 8
## Donc je retranche 1 aux valeurs de mes labels
emotion_train = emotion_train - 1
emotion_test = emotion_test - 1

print(np.unique(emotion_train))
print(np.unique(emotion_test))

[0 1 2 3 4 5 6 7]
[0 1 2 3 4 5 6 7]


In [ ]:
model = models.Sequential()
## je cree ma premiere couche de convolution
## Je cree une couche de convoltion de 32 filtres, d'un kernel de dimension (3,3) pour des images de tailles 64x64x1 
## Le padding représente la dimension de ma matrice de sortie, juste apres convoltion. En mettant padding="same" on concerve le max d'informations
model.add(layers.Conv2D(32,(3,3),activation="relu",padding='same',input_shape = (40,350,1)))

## Le MaxPooling utilise une fenetre de 2x2 pour résuire la dimension du résultat de la convolution
## Après le MaxPooling2D on aura une matrice de taille/2 c'est a dire 350/2=175 et 40/2=20 et de profondeur 32 filtres
model.add(layers.MaxPooling2D((2,2)))

model.add(layers.Conv2D(64,(3,3),activation='relu',padding='same'))
## La on applique 64 filtres et on force le resultat a etre de la même dimension que l'entrée
## On aura donc une matrice de taille 175x20 et de profondeur 64
## Chaque nouveau filtres utilise toutes les maps precedentes pour produires une seule map

model.add(layers.MaxPooling2D(2,2))
## La on a des matrices de tailles 175//2 = 87 x 20/2=10 et de profondeur 64



model.add(layers.Conv2D(128,(3,3),activation='relu',padding='same'))
model.add(layers.MaxPooling2D(2,2))
## ma taille est de 87//2=43 x 5 et de profondeur 128



## J'applaties maintenant mes données pour avoir un seul vecteur que je vais faire passer dans un réseau de neurones profonds
model.add(layers.Flatten())


## je cree mon reseau de neurones profonds
model.add(layers.Dense(128,activation='relu'))
model.add(layers.Dropout(0.4))
model.add(layers.Dense(8,"softmax"))


In [144]:
## L'entrainemet
model.compile(optimizer='adam',## optimizer represente l'algorithme utilisé pour mettre a jour les poids, par ex descente de gradient, ect
              loss = 'sparse_categorical_crossentropy', ## la fonction qui calcule le score ou l'erreur
              metrics = ['accuracy'])## Permet a l'humain de voir l'exactitude de ces résultats

In [158]:
history1 = model.fit(mfcc_train,emotion_train,validation_data=(mfcc_test,emotion_test),epochs=20,batch_size=32)

Epoch 1/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 7s 172ms/step - accuracy: 0.4800 - loss: 1.2988 - val_accuracy: 0.3375 - val_loss: 1.9650
Epoch 2/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 140ms/step - accuracy: 0.4833 - loss: 1.2968 - val_accuracy: 0.3375 - val_loss: 1.9626
Epoch 3/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 6s 161ms/step - accuracy: 0.4858 - loss: 1.2998 - val_accuracy: 0.3125 - val_loss: 2.0138
Epoch 4/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 6s 157ms/step - accuracy: 0.4858 - loss: 1.2883 - val_accuracy: 0.3500 - val_loss: 1.9361
Epoch 5/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 6s 169ms/step - accuracy: 0.4775 - loss: 1.3394 - val_accuracy: 0.3458 - val_loss: 2.0592
Epoch 6/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 6s 170ms/step - accuracy: 0.4867 - loss: 1.2915 - val_accuracy: 0.3167 - val_loss: 1.9573
Epoch 7/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 6s 167ms/step - accuracy: 0.4883 - loss: 1.2858 - val_accuracy: 0.3042 - val_loss: 2.1133
Epoch 8/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 7s 172ms/step - accuracy: 0.4850 - loss: 1.2684 - val_accuracy: 0.

In [148]:
model2 = models.Sequential()
model2.add(layers.Conv1D(filters=32,kernel_size=3,activation='relu',input_shape =(350,40),padding = 'same' ))
model2.add(layers.MaxPooling1D(pool_size=2))

model2.add(layers.Conv1D(filters=64,
                        kernel_size=3,
                        activation='relu',
                        padding = 'same' ))
model2.add(layers.MaxPool1D(pool_size=2))

model2.add(layers.Conv1D(128,kernel_size=3,activation='relu',padding="same"))
model2.add(layers.MaxPooling1D(pool_size=2))


# Au lieu de Flatten, on fait la moyenne temporelle de chaque filtre. 
# C'est beaucoup plus robuste face aux variations de durée des audios !
model2.add(layers.GlobalAveragePooling1D())


model2.add(layers.Dense(64,activation='relu'))
model2.add(layers.Dropout(0.4))
model2.add(layers.Dense(8,activation='softmax'))

model2.summary()


c:\Users\sanou\OneDrive\Documents(1)\CODE_ALPHA\Emotion recognition\mon_environnement\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_15"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_3 (Conv1D)               │ (None, 350, 32)        │         3,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ (None, 175, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_4 (Conv1D)               │ (None, 175, 64)        │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_4 (MaxPooling1D)  │ (None, 87, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_5 (Conv1D)               │ (None, 87, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_5 (MaxPooling1D)  │ (None, 43, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_37 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_38 (Dense)                │ (None, 8)              │           520 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 43,560 (170.16 KB)

 Trainable params: 43,560 (170.16 KB)

 Non-trainable params: 0 (0.00 B)

In [149]:
## Je dois réorganiser mes données pour les faire entrer dans conv1d
mfcc_train_1d = mfcc_train.reshape(1200,350,40)
mfcc_test_1d = mfcc_test.reshape(240,350,40)

In [ ]:
model2.compile(optimizer='adam',
                 loss='sparse_categorical_crossentropy',
                 metrics=['accuracy'])

In [160]:
history2 = model2.fit(mfcc_train_1d, emotion_train,validation_data=(mfcc_test_1d, emotion_test),epochs=40,batch_size=32)

Epoch 1/40
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.4067 - loss: 1.5468 - val_accuracy: 0.4125 - val_loss: 1.5784
Epoch 2/40
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4092 - loss: 1.5323 - val_accuracy: 0.3625 - val_loss: 1.6006
Epoch 3/40
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4000 - loss: 1.5415 - val_accuracy: 0.3708 - val_loss: 1.5877
Epoch 4/40
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4017 - loss: 1.5596 - val_accuracy: 0.4208 - val_loss: 1.5867
Epoch 5/40
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4067 - loss: 1.5349 - val_accuracy: 0.4042 - val_loss: 1.6283
Epoch 6/40
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4025 - loss: 1.5213 - val_accuracy: 0.3875 - val_loss: 1.5699
Epoch 7/40
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4100 - loss: 1.5402 - val_accuracy: 0.4000 - val_loss: 1.6138
Epoch 8/40
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3983 - loss: 1.5383 - val_accuracy: 0.3833 - val_loss